In [1]:
# !pip install accelerate rouge kiwipiepy lm-eval

In [2]:
# 정량적 평가 BLEU(기계번역 성능평가)  ROUGE
# BLEU 예측문장과 정답문장이 얼마나 많은 n-gram을 공유
# BLUE-1 : unigram
# 단점 : 동의어를 인식 못함  The car is fast  /  The automobile is quick

# ROUGE(문서요약 평가)
# n-gram을 비교.. Recall 중심  The cat sat on the mat / The cat sat
# 정답 6, 예측 3   6/3 = 0.5 -- recall    3 / 3 ->1.0  Precision
# ROUGE-1 : unigram..

# 최근에는 다양한 평가지표가 사용 BERTScore...LLM-as-a-juge

In [17]:
from nltk.translate.bleu_score import sentence_bleu,SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram_improved(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    blue_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    return blue_score, scores

ref = 'When traveling to Paris, you should definitely visit the Eiffel Tower and the Louvre Museum.'
cand = 'The Eiffel Tower and the Louvre Museum are must-see attractions when visiting Paris.'

bleu, rouge_rs = evaluate_ngram_improved(ref,cand)

print(f'BLUE : {bleu}')
print(f"ROUGE-1 F1: {rouge_rs['rouge-1']['f']:.4f}")

BLUE : 0.21530464891161827
ROUGE-1 F1: 0.3871


In [18]:
from lm_eval.tasks import TaskManager
task_manager = TaskManager()
all_tasks = list(task_manager.task_index.keys())
all_tasks

['aclue',
 'aclue_ancient_chinese_culture',
 'aclue_ancient_literature',
 'aclue_ancient_medical',
 'aclue_ancient_phonetics',
 'aclue_basic_ancient_chinese',
 'aclue_couplet_prediction',
 'aclue_homographic_character_resolution',
 'aclue_named_entity_recognition',
 'aclue_poetry_appreciate',
 'aclue_poetry_context_prediction',
 'aclue_poetry_quality_assessment',
 'aclue_poetry_sentiment_analysis',
 'aclue_polysemy_resolution',
 'aclue_reading_comprehension',
 'aclue_sentence_segmentation',
 'acp_areach_bool',
 'acp_bool_cot_2shot',
 'acp_bench',
 'acp_app_bool',
 'acp_just_bool',
 'acp_land_bool',
 'acp_prog_bool',
 'acp_reach_bool',
 'acp_val_bool',
 'acp_areach_gen',
 'acp_gen_2shot',
 'acp_bench_hard',
 'acp_app_gen',
 'acp_just_gen',
 'acp_land_gen',
 'acp_nexta_gen',
 'acp_prog_gen',
 'acp_reach_gen',
 'acp_val_gen',
 'acp_areach_gen_with_pddl',
 'acp_gen_2shot_with_pddl',
 'acp_bench_hard_with_pddl',
 'acp_app_gen_with_pddl',
 'acp_just_gen_with_pddl',
 'acp_land_gen_with_pddl',

In [19]:
[task for task in all_tasks if 'hellaswag' in task]

['arabic_leaderboard_arabic_mt_hellaswag',
 'arabic_mt_hellaswag',
 'arabic_leaderboard_arabic_mt_hellaswag_light',
 'arabic_mt_hellaswag_light',
 'darijahellaswag',
 'egyhellaswag',
 'french_bench_hellaswag',
 'hellaswag',
 'kobest_hellaswag',
 'metabench_hellaswag',
 'metabench_hellaswag_subset',
 'metabench_hellaswag_permute',
 'metabench_hellaswag_secondary',
 'metabench_hellaswag_secondary_permute',
 'hellaswag_ar',
 'hellaswag_multilingual',
 'hellaswag_bn',
 'hellaswag_ca',
 'hellaswag_da',
 'hellaswag_de',
 'hellaswag_es',
 'hellaswag_eu',
 'hellaswag_fr',
 'hellaswag_gu',
 'hellaswag_hi',
 'hellaswag_hr',
 'hellaswag_hu',
 'hellaswag_hy',
 'hellaswag_id',
 'hellaswag_it',
 'hellaswag_kn',
 'hellaswag_ml',
 'hellaswag_mr',
 'hellaswag_ne',
 'hellaswag_nl',
 'hellaswag_pt',
 'hellaswag_ro',
 'hellaswag_ru',
 'hellaswag_sk',
 'hellaswag_sr',
 'hellaswag_sv',
 'hellaswag_ta',
 'hellaswag_te',
 'hellaswag_uk',
 'hellaswag_vi']

In [20]:
# LM-Evaluation_Harness벤치 마크 평가
# 모델의 추론 능력을 벤치마트 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
try:
    result = lm_eval.simple_evaluate(
        model="hf",
        model_args = "pretrained=skt/kogpt2-base-v2,dtype=float32",
        tasks=['kobest_hellaswag'],
        device="cpu",
        limit=2
    )
    print(f"LM-Eval 정확도(acc): {result['results']['hellaswag']['acc,none']}")
except Exception as e:
    print(e)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


README.md:   0%|          | 0.00/7.20k [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--skt--kobest_v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.jsonl:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2029 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:01<00:00,  5.02it/s]


'hellaswag'


In [22]:
result['results']['kobest_hellaswag']

{'name': 'kobest_hellaswag',
 'alias': 'kobest_hellaswag',
 'sample_len': 2,
 'acc,none': 0.0,
 'acc_stderr,none': 0.0,
 'f1,none': 0.0,
 'f1_stderr,none': 'N/A',
 'acc_norm,none': 0.0,
 'acc_norm_stderr,none': 0.0}

In [16]:
# 정답형태가 고정되어 있지않은 생성형 태스크의 경우 상용모델(gpt)을 판단기준으로 활용
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()
def evaluate_with_gpt(prompt, generated_text):
    eval_prompt = f'''
다음 질문과 답변을 보고 정확성, 유창성, 관련성을 기준으로 1~5점 사이의 점수와 이류를 작성해주세요
질문:{prompt}
답변:{generated_text}
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages=[{'role':'user','content':eval_prompt}],
            max_completion_tokens = 150
        )
        return response.choices[0].message.content
    except Exception as e:
        return e
print(evaluate_with_gpt('프랑시의 명소는?','파리 여행시 에팔탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.')    )
    

- **정확성: 3/5**  
  - 에펠탑과 루브르 박물관은 프랑스(특히 파리)의 대표적인 명소로 맞습니다.  
  - 다만 질문이 “프랑시의 명소는?”으로 넓게 되어 있어, 답변이 파리 중심으로만 제시되어 **프랑스 전반의 명소**라고 보기엔 범위가 좁습니다.

- **유창성: 4/5**  
  - 문장이 자연스럽고 의미 전달이 잘 됩니다.  
  - 다만 “에팔탑”은 보통 “에펠탑”으로 표기
